# The LLM Sandbox: Cloud Training Notebook
Run this notebook on Google Colab or Kaggle (with a T4 GPU) to fine-tune your model using LoRA.

## 1. Installation and Imports

In [ ]:
!pip install torch transformers tiktoken matplotlib

import os
import sys
# Add src to path
sys.path.append(os.path.abspath(".."))

import json
import torch
import tiktoken
from torch.utils.data import DataLoader

from src.model import GPTModel
from src.weights import download_and_load_gpt2
from src.lora import replace_linear_with_lora
from src.dataset import InstructionDataset, custom_collate_fn, format_input
from src.training import train_model

## 2. Model Configuration & Loading Pre-trained Weights

In [ ]:
GPT_CONFIG_355M = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 1024,
    "n_heads": 16,
    "n_layers": 24,
    "drop_rate": 0.0,
    "qkv_bias": True
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Instantiate our custom architecture
model = GPTModel(GPT_CONFIG_355M)
model.eval()

# Download and map OpenAI weights (This requires internet access)
download_and_load_gpt2(model, "355M")
model.to(device);

## 3. Injecting LoRA (Low-Rank Adaptation)

In [ ]:
import time

# Freeze all base model parameters
for param in model.parameters():
    param.requires_grad = False

# Replace linear layers with LoRA layers
replace_linear_with_lora(model, rank=16, alpha=16)

model.to(device)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total params: {total_params:,}")
print(f"Trainable params (LoRA): {trainable_params:,} ({trainable_params/total_params*100:.2f}%)")

## 4. Dataset Preparation

In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")

# Load JSON data
with open("../data/instruction-data.json", "r") as f:
    data = json.load(f)

# Split into train and validation sets
train_portion = int(len(data) * 0.85)
train_data = data[:train_portion]
val_data = data[train_portion:]

import functools
customized_collate_fn = functools.partial(
    custom_collate_fn, device=device, allowed_max_length=1024
)

train_dataset = InstructionDataset(train_data, tokenizer)
val_dataset = InstructionDataset(val_data, tokenizer)

train_loader = DataLoader(
    train_dataset, batch_size=8, collate_fn=customized_collate_fn, shuffle=True, drop_last=True
)
val_loader = DataLoader(
    val_dataset, batch_size=8, collate_fn=customized_collate_fn, shuffle=False, drop_last=False
)

## 5. Fine-Tuning the Model

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.1)

# Format a sample prompt to monitor generation quality during training
sample_entry = val_data[0]
start_context = format_input(sample_entry)

start_time = time.time()
train_losses, val_losses, tokens_seen, lrs = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=device,
    n_epochs=2,
    eval_freq=10,
    eval_iter=5,
    start_context=start_context,
    tokenizer=tokenizer,
    warmup_steps=10,
    initial_lr=1e-5,
    min_lr=1e-5
)
end_time = time.time()
print(f"Training completed in {(end_time - start_time)/60:.2f} minutes.")

## 6. Save the LoRA Weights
We save only the LoRA weights, not the entire 355M model, making the resulting file extremely small (~10MB).

In [ ]:
lora_state_dict = {k: v for k, v in model.state_dict().items() if "lora" in k}
torch.save(lora_state_dict, "lora_weights.pth")
print("Saved lora_weights.pth! Download this file to your local computer.")